# Lecture 16 - Exceptions and Unit Testing (https://bit.ly/intro_python_16)

* Unit testing with unittest
* Exceptions and error handling
* Putting the two together: testing that your code fails in the way it should

# Why this lecture matters: a 370 million dollar exception

On 4 June 1996 the first Ariane 5 rocket veered off course and destroyed itself about 40 seconds after launch.

The cause was a piece of software, reused from the earlier Ariane 4, that converted a 64-bit floating point number into a 16-bit integer. Ariane 5 was faster than Ariane 4, the number was too big to fit, the code raised an exception, and nothing was in place to handle it. The rocket and its four satellites, worth about 370 million dollars, were lost. (https://en.wikipedia.org/wiki/Ariane_flight_V88)

There are two lessons in the story, and they are today's two topics:

* **Testing:** the code had never been run with the numbers the new rocket would produce.
* **Exceptions:** when something unexpected happens in a running program, you want to have decided in advance what should happen next.

# Unit testing with the unittest module

* As programs grow in complexity, the scope for bugs becomes huge.

* Satisfactorily debugging a complex program without systematic testing is **hard or even intractable**.

* With **unit-testing** you design tests to test individual "units" of the code, e.g. functions and classes.

* Unit-testing allows you to progressively debug code and build tested modules of code with less fear that nothing will work when you finally put it together.

# Unit testing example: Fibonacci numbers

* Let's look at a simple example, debugging a function for computing members of the Fibonacci sequence, which we met last lecture.

* Recall the ith Fibonacci number is equal to the sum of the previous two Fibonacci numbers, and 0 and 1 are the 0th and 1st Fibonacci numbers.

  * i.e. fib(i) = fib(i-1) + fib(i-2), for i > 1

* This definition is naturally recursive, so we can use a recursive implementation of the function.

* Here is an attempt. **Is it correct?** Read it carefully before we test it.

In [1]:
def fib(i):
  """ Compute the ith fibonacci number recursively
  """
  assert type(i) == int and i >= 0
  if i == 0:
    return 1
  if i == 1:
    return 1
  return fib(i-1) + fib(i-2)

You already know one way to test a function. The `assert` lines at the bottom of nearly every challenge this quarter have been mini tests:

In [2]:
assert fib(1) == 1
assert fib(0) == 0
assert fib(6) == 8

AssertionError: 

So there is a bug. But plain asserts make poor tests: we aren't told what `fib(0)` actually returned, and everything after the first failure never gets checked.

The `unittest` module does the same job in a more organized way:

In [3]:
import unittest  # unittest is the standard Python library module for unit testing,
# it's great

class FibTest(unittest.TestCase):  # Note the use of inheritance
  """ We create a test class that inherits from the
  unittest.TestCase class
  """
  def test_fib(self):
    """ Each test we create must start with the name "test"
    """
    self.assertEqual(fib(0), 0)
    self.assertEqual(fib(1), 1)
    self.assertEqual(fib(2), 1)
    self.assertEqual(fib(3), 2)
    self.assertEqual(fib(4), 3)
    self.assertEqual(fib(5), 5)
    self.assertEqual(fib(6), 8)

unittest.main(argv=['first-arg-is-ignored'], exit=False)

F
FAIL: test_fib (__main__.FibTest.test_fib)
Each test we create must start with the name "test"
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/tmp/ipykernel_337/4149540761.py", line 11, in test_fib
    self.assertEqual(fib(0), 0)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^
AssertionError: 1 != 0

----------------------------------------------------------------------
Ran 1 test in 0.001s

FAILED (failures=1)


## Reading the output

* The first line has one character per test: `.` for a pass, `F` for a failure and `E` for an error (more on the difference later).
* For each failure you get the name of the test, the line that failed, and what went wrong. `AssertionError: 1 != 0` says that `fib(0)` returned 1 and we expected 0.
* The last lines give the number of tests run and the overall result.

A note on that last line of code. In a normal Python file you would just write `unittest.main()`. The two arguments are only needed because we are inside a notebook: `argv=[...]` stops unittest from trying to read the notebook's own command line arguments, and `exit=False` stops it from shutting down Python when the tests finish.

Okay, so our test failed. Let's fix the function:

In [4]:
def fib(i):
  """ Compute the ith fibonacci number recursively

  Reminder the ith fibonacci number is equal to the sum of the
  previous two fibonacci numbers, where the 0th fibonacci
  number is 0 and the 1st is 1.
  """
  assert type(i) == int and i >= 0
  if i == 0:  # Fixed!
    return 0
  if i == 1:
    return 1
  return fib(i-1) + fib(i-2)

Now we can rerun the tests:

In [5]:
class FibTest(unittest.TestCase):  # Note the use of inheritance
  """ We create a test class that inherits from the
  unittest.TestCase class
  """
  def test_fib(self):
    """ IMPORTANT NOTE: Each test we create must start with the name "test",
    otherwise it will not be considered a test but just a helper method
    """
    self.assertEqual(fib(0), 0)
    self.assertEqual(fib(1), 1)
    self.assertEqual(fib(2), 1)
    self.assertEqual(fib(3), 2)
    self.assertEqual(fib(4), 3)
    self.assertEqual(fib(5), 5)
    self.assertEqual(fib(6), 8)

unittest.main(argv=['first-arg-is-ignored'], exit=False)

.
----------------------------------------------------------------------
Ran 1 test in 0.001s

OK


Okay, this example is contrived, but the idea that you should write code to test your code turns out to be remarkably useful. The tests stay with the program, so every time you change `fib` in future you can rerun them and find out within a second whether you broke anything.

# More ways to assert

`unittest.TestCase` gives you a family of assert methods. Using the most specific one gets you the most helpful message when a test fails.

| Method | Passes if |
|---|---|
| `self.assertEqual(a, b)` | `a == b` |
| `self.assertNotEqual(a, b)` | `a != b` |
| `self.assertTrue(x)` | `x` is true |
| `self.assertFalse(x)` | `x` is false |
| `self.assertIn(item, collection)` | `item in collection` |
| `self.assertAlmostEqual(a, b)` | `a` and `b` are equal to 7 decimal places |
| `self.assertRaises(SomeError)` | the code raises `SomeError` (we'll use this at the end of the lecture) |

Why would you need `assertAlmostEqual`? Predict what this prints:

In [6]:
print(0.1 + 0.2 == 0.3)
print(0.1 + 0.2)

False
0.30000000000000004


Computers store decimal fractions in binary, and 0.1 has no exact binary representation, just as 1/3 has no exact decimal one. The tiny rounding errors mean you should never test floating point numbers with `assertEqual`; use `assertAlmostEqual`.

# Writing multiple tests: setUp() and tearDown()

What if you want to write multiple tests for multiple different functions?

* It is good to keep tests small and modular: one test method for each behavior you care about, with a name that says what it checks.

* If you want to test lots of related functions, e.g. the methods of a class, it is therefore helpful to have shared "setup" and "cleanup" functions that are run, respectively, before and after each test. You can achieve this with the *setUp()* and *tearDown()* methods of unittest.TestCase.

In [7]:
# Here's the card class we studied before when looking at the Old Maid card game:

class Card:
  """ Represents a card from a deck of cards. """

  # These are class variables: shared by all cards
  suits = ["Clubs", "Diamonds", "Spades", "Hearts"]
  ranks = ["narf", "Ace", "2", "3", "4", "5", "6", "7",
           "8", "9", "10", "Jack", "Queen", "King"]

  def __init__(self, suit=0, rank=0):
    """ Create a card using integers to represent suit and rank. """
    # A couple of handy asserts to check any cards we build make sense
    assert 0 <= suit < 4
    assert 0 <= rank < 14

    self.suit = suit
    self.rank = rank

  def __str__(self):
    # Looking up the suits/ranks lists gives a human readable card
    return self.ranks[self.rank] + " of " + self.suits[self.suit]

  def __repr__(self):
    return str(self)

  def same_color(self, other):
    """ True if the cards have the same color.
    Diamonds and Hearts are both red, Clubs and Spades are both black.
    """
    return self.suit == other.suit or self.suit == (other.suit + 2) % 4

  # Comparison: cards are compared first by suit, then by rank

  def sort_key(self):
    return (self.suit, self.rank)

  def __eq__(self, other):
    if not isinstance(other, Card):
      return NotImplemented
    return self.sort_key() == other.sort_key()

  def __lt__(self, other):
    return self.sort_key() < other.sort_key()

  def __le__(self, other):
    return self.sort_key() <= other.sort_key()

  def __gt__(self, other):
    return self.sort_key() > other.sort_key()

  def __ge__(self, other):
    return self.sort_key() >= other.sort_key()

To test the individual methods we could do something like this:

In [8]:
class CardTest(unittest.TestCase):
  """ Test the Card class """

  def setUp(self):
    print("setUp")
    # This function gets run before each test
    self.aceClubs = Card(0, 1)     # Ace of clubs
    self.aceDiamonds = Card(1, 1)  # Ace of diamonds
    self.aceSpades = Card(2, 1)    # Ace of spades

  def tearDown(self):
    # This function gets run after each test.
    # Here I do nothing in teardown except print a message,
    # but you can use it to clean up temporary files, etc.
    print("tearDown")

  def test_same_color(self):
    """ Tests Card.same_color() """
    print("Running test_same_color")  # These print messages are just to show you what's going on
    self.assertTrue(self.aceClubs.same_color(self.aceSpades))
    self.assertFalse(self.aceSpades.same_color(self.aceDiamonds))

  def test_str(self):
    """ Tests Card.__str__()"""
    print("Running test_str")
    self.assertEqual(str(self.aceClubs), "Ace of Clubs")
    self.assertEqual(str(self.aceSpades), "Ace of Spades")
    self.assertEqual(str(self.aceDiamonds), "Ace of Diamonds")

unittest.main(argv=['first-arg-is-ignored'], exit=False)

...
----------------------------------------------------------------------
Ran 3 tests in 0.002s

OK


setUp
Running test_same_color
tearDown
setUp
Running test_str
tearDown


Things to notice in the output:

* `setUp` and `tearDown` ran once around *each* test, so every test starts with a fresh set of cards and tests cannot interfere with each other.
* It says three tests were run, although CardTest only has two. `unittest.main` runs every test class defined so far in the notebook, so FibTest ran again as well.
* To run a single test class, add its name to the list: `unittest.main(argv=['first-arg-is-ignored', 'CardTest'], exit=False)`

# Challenge 1

In [ ]:
# Create your own CardTest unit test class, called ExpandedCardTest, which additionally
# includes a test for
# comparing cards using the equals and comparison operators. The test should compare
# the expected ordering of the three cards created by the setUp method
# (ace of clubs, ace of diamonds, ace of spades)
# In this program "Clubs" < "Diamonds" < "Spades" < "Hearts".

# Run your tests!

# What makes a good test? A bug hunt

A test that only tries the obvious input mostly confirms what you already believed. The useful tests are the ones that try to break the code:

* A typical case, with an answer you worked out by hand
* **Edge cases:** zero, one item, an empty string or list, negative numbers, very large values, duplicates, upper and lower case
* Inputs that should be rejected

Here is a function that calculates the fraction of a DNA sequence that is G or C. It looks reasonable, and `gc_content("GATTACA")` gives the right answer. Before running the tests, can you think of two inputs that would break it?

In [9]:
def gc_content(seq):
  """ Returns the fraction of the bases in a DNA sequence that are G or C """
  return (seq.count("G") + seq.count("C")) / len(seq)

gc_content("GATTACA")

0.2857142857142857

In [10]:
class GCContentTest(unittest.TestCase):
  """ Tests gc_content() """

  def test_typical(self):
    self.assertAlmostEqual(gc_content("GGCC"), 1.0)
    self.assertAlmostEqual(gc_content("ATAT"), 0.0)
    self.assertAlmostEqual(gc_content("GATTACA"), 2/7)

  def test_lowercase(self):
    self.assertAlmostEqual(gc_content("ggcc"), 1.0)

  def test_empty(self):
    self.assertEqual(gc_content(""), 0)

unittest.main(argv=['first-arg-is-ignored', 'GCContentTest'], exit=False)  # Run only GCContentTest

EF.
ERROR: test_empty (__main__.GCContentTest.test_empty)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/tmp/ipykernel_337/2182261924.py", line 13, in test_empty
    self.assertEqual(gc_content(""), 0)
                     ~~~~~~~~~~^^^^
  File "/tmp/ipykernel_337/1351771665.py", line 3, in gc_content
    return (seq.count("G") + seq.count("C")) / len(seq)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^~~~~~~~~~
ZeroDivisionError: division by zero

FAIL: test_lowercase (__main__.GCContentTest.test_lowercase)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/tmp/ipykernel_337/2182261924.py", line 10, in test_lowercase
    self.assertAlmostEqual(gc_content("ggcc"), 1.0)
    ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: 0.0 != 1.0 within 7 places (1.0 difference)

----------------------------------------------------------------------
Ran 3 t

Two different kinds of problem are reported:

* `F`, a **failure**: the function ran, but an assertion did not hold. Lowercase sequences give the wrong answer, 0.0 where we expected 1.0.
* `E`, an **error**: the code crashed before the assertion could even be checked. An empty sequence causes a `ZeroDivisionError`.

Let's fix both and rerun the same tests:

In [11]:
def gc_content(seq):
  """ Returns the fraction of the bases in a DNA sequence that are G or C """
  if len(seq) == 0:
    return 0
  seq = seq.upper()
  return (seq.count("G") + seq.count("C")) / len(seq)

unittest.main(argv=['first-arg-is-ignored', 'GCContentTest'], exit=False)

...
----------------------------------------------------------------------
Ran 3 tests in 0.002s

OK


Is returning 0 for an empty sequence the right decision? And what should `gc_content("GATXACA")` do? Those are design questions rather than bugs, and we'll come back to them once we know how to raise exceptions of our own.

# Writing test suites

* Generally for each Python module you write you create an accompanying unittest module.
  * e.g. if you write "foo.py" you also create "fooTest.py".

* It's beyond scope here, but as you write more complex programs, with multiple modules organized into packages, you can automate running all your tests together in one test suite. When you make a change to your code you then rerun all the tests and check everything is still good.

* As a rough rule of thumb, good programmers write about as much unit test code as they write program code.
  * It seems like a long way around, but it is generally quicker and more manageable than the ad hoc debugging which is otherwise inevitable.

* One popular approach is to write the tests before writing the core of the program. This is part of the philosophy of "test driven development".
  * This helps figure out what the program should do and how it should behave before going too far into the actual implementation.

* When you find a bug, write a test that fails because of it *before* you fix it. The bug can then never quietly come back.

* Outside this course you will often meet pytest, a popular third-party testing tool in which tests are plain functions containing `assert` statements. The ideas are the same.

# Exceptions

When a runtime error occurs Python creates an exception. We've seen these, e.g.:

In [12]:
assert False  # Creates an AssertionError, a kind of exception

AssertionError: 

The `E` in our bug hunt was another, a `ZeroDivisionError`. Here it is again in a slightly bigger program:

In [13]:
def mean(numbers):
  return sum(numbers) / len(numbers)

def report(samples):
  for name in samples:
    print(name, "mean:", mean(samples[name]))

report({"control": [4.1, 3.9, 4.4], "treated": []})

control mean: 4.133333333333334


ZeroDivisionError: division by zero

## How to read a traceback

The message Python prints when an exception is not handled is called a traceback. It is worth learning to read rather than skipping over:

1. **Start at the bottom.** The last line gives the type of exception and a message: `ZeroDivisionError: division by zero`.
2. **Then read upwards.** Each block is one function call, with an arrow pointing at the line that was running. The bottom block is where the error happened (inside `mean`). The blocks above show how the program got there (`report` called `mean`, and our cell called `report`).

So the traceback tells a story: we called `report`, which got as far as the "treated" sample, called `mean` with an empty list, and divided by zero.

# Handling exceptions: try / except

* So far we've just encountered exceptions when the program fails, but actually we can frequently handle exceptions within the program and not crash.

* To do this we use the try / except syntax:

```python
try:
  STATEMENT_BLOCK_1
except ERROR_TYPE:
  STATEMENT_BLOCK_2
```

The way this works:

* The statement block STATEMENT_BLOCK_1 is executed.
* If an exception of type ERROR_TYPE occurs during the execution of STATEMENT_BLOCK_1 then STATEMENT_BLOCK_1 stops immediately and STATEMENT_BLOCK_2 executes.
* If no exception occurs during STATEMENT_BLOCK_1, STATEMENT_BLOCK_2 is skipped.
* This allows us to handle unexpected events in a predictable way.

Converting text to numbers is a common place for things to go wrong:

In [14]:
print(int("42"))
print(int("forty-two"))

42


ValueError: invalid literal for int() with base 10: 'forty-two'

In [15]:
try:
  number = int("forty-two")
  print("This line is skipped, because the line above raised an exception")
except ValueError:
  print("That wasn't a valid integer")

print("...and the program carries on")

That wasn't a valid integer
...and the program carries on


This is very useful with real data, which is rarely as clean as you would like. Here we add up the readings that are valid numbers and keep a note of the ones that are not, without the loop stopping:

In [16]:
readings = ["12", "7", "n/a", "3", "", "42", "1O"]  # look closely at the last one

total = 0
rejected = []
for text in readings:
  try:
    total += int(text)
  except ValueError:
    rejected.append(text)

print("Total of the valid readings:", total)
print("Could not convert:", rejected)

Total of the valid readings: 64
Could not convert: ['n/a', '', '1O']


User input is another classic source of errors:

In [17]:
i = int(input("Enter an integer: "))  # What happens if I don't enter a valid integer?

Enter an integer: hello


ValueError: invalid literal for int() with base 10: 'hello'

We can handle this using try/except, and keep asking until we get something usable:

In [18]:
while True:
  try:
    i = int(input("Enter an integer: "))
    break
  except ValueError:
    print("Got an error parsing user input, try again!")

print("You entered: ", i)

Enter an integer: ten
Got an error parsing user input, try again!
Enter an integer: 10.5
Got an error parsing user input, try again!
Enter an integer: 10
You entered:  10


**You don't have to specify the exception type**

* If you don't know what error to anticipate you can leave out the type of exception:

In [19]:
while True:
  try:
    i = int(input("Enter an integer: "))
    break
  except:  # Note we don't say what kind of error it is
    print("Got an error parsing user input, try again!")

print("You entered: ", i)

Enter an integer: 3.5
Got an error parsing user input, try again!
Enter an integer: 3
You entered:  3


The downside of not specifying the type of the expected exception is that an except without a type will catch all exceptions, including errors you never anticipated. That can hide bugs. Can you see what has gone wrong here?

In [20]:
readings = ["12", "7", "42"]

total = 0
for text in readings:
  try:
    total += int(txt)
  except:
    print("Not a valid integer:", text)

print("Total:", total)

Not a valid integer: 12
Not a valid integer: 7
Not a valid integer: 42
Total: 0


Every reading was valid. The real problem is a typo, `txt` instead of `text`, which raises a `NameError` each time around the loop. The bare `except:` caught it and reported something misleading. Had we written `except ValueError:`, Python would have shown us the NameError straight away.

**Catch the most specific exception you can.**

To handle multiple types of exception you can have multiple except clauses:

In [21]:
while True:
  try:
    i = int(input("Enter an integer greater than 10: "))
    assert i > 10  # Number should be > 10
    break
  except ValueError:
    print("Not a valid integer, try again!")
  except AssertionError:
    print("Number is not greater than 10, try again")

print("You entered: ", i)

Enter an integer greater than 10: five
Not a valid integer, try again!
Enter an integer greater than 10: 5
Number is not greater than 10, try again
Enter an integer greater than 10: 50
You entered:  50


In [22]:
# Note this is not valid python because a catchall except statement must be at the end:

while True:
  try:
    i = int(input("Enter an integer greater than 10: "))
    assert i > 10  # Number should be > 10
    break
  except:
    print("Not a valid integer, try again!")
  except AssertionError:
    print("Number is less than 10, try again")

print("You entered: ", i)

SyntaxError: default 'except:' must be last (3875544046.py, line 8)

If you want to treat several kinds of exception the same way, list them in brackets:

```python
except (ValueError, AssertionError):
  print("Please enter an integer greater than 10")
```

And recall from Lecture 14 that exceptions form a family tree, so catching a parent class catches its children too. `except ArithmeticError:` will catch a `ZeroDivisionError`, and `except Exception:` catches nearly everything.

You can name the exception you catch with the "as" keyword. The exception is an object, and it carries the error message with it:

In [23]:
try:
  assert False
except AssertionError as e:
  print("Got an assert error", type(e))

try:
  int("forty-two")
except ValueError as e:
  print("Got a value error with the message:", e)

Got an assert error <class 'AssertionError'>
Got a value error with the message: invalid literal for int() with base 10: 'forty-two'


# Challenge 2

In [ ]:
# Practice problem

# Write a function "get_file" to ask the user for a file name.
# Return an open file handle to the users file in "read" mode.
# Use exception handling to deal with the case that the user's file
# does not exist, printing an error saying "File does not exist, try again"
# and trying again to get a file from the user name

# Hint use FileNotFoundError


## Use this code to test your method
import os
with open("test.txt", "w") as f:  ## Make a file to open if you don't have one in your working directory
  f.write("Putting some text in the file")
print("Files you could enter", os.listdir())
f = get_file()  # Ask the user for the file
print("Contents of user file", f.read())
f.close()

# Finally and else with exceptions

Finally allows us to specify code that will be run regardless of whether there is an error:

In [26]:
try:
  f = open("out.txt", "w")
  f.write("Hello, file!\n")
  assert 1 == 2
except:
  print("Got an error")
finally:
  print("Closing the file")
  f.close()

Got an error
Closing the file


The way this works:
* if there is an error, the error is dealt with.
* the finally clause is then run, regardless of whether there was an error or not

This is what the `with` statement has been doing for you since the lecture on files. `with open(...) as f:` closes the file when the block ends, even if an exception is raised inside it, so for files you rarely need to write `finally` yourself.

The full form of the statement has four parts. An `else` block, if you include one, runs only when no exception occurred. Predict what each of these two calls prints, and in what order:

In [27]:
def divide(a, b):
  try:
    result = a / b
  except ZeroDivisionError:
    print("  except: can't divide by zero")
    return None
  else:
    print("  else: no exception happened")
    return result
  finally:
    print("  finally: this always runs, even though we are returning")

print("divide(6, 3):")
print("  result is", divide(6, 3))

print("divide(6, 0):")
print("  result is", divide(6, 0))

divide(6, 3):
  else: no exception happened
  finally: this always runs, even though we are returning
  result is 2.0
divide(6, 0):
  except: can't divide by zero
  finally: this always runs, even though we are returning
  result is None


# Raise

So far Python has been raising the exceptions. If you want to signal an error yourself, use "raise":

In [28]:
def get_age():
  """Returns a user's age as an integer"""
  age = int(input("Please enter your age: "))
  if age < 0:
    # Create a new instance of an exception
    my_error = ValueError(f"{age} is not a valid age")
    raise my_error
  return age

In [29]:
get_age()

Please enter your age: -5


ValueError: -5 is not a valid age

An exception is an object, made by calling its class like any other. The usual way to write it is in one line:

```python
raise ValueError(f"{age} is not a valid age")
```

Raising an exception is how a function says "I can't do what you asked, and here is why". The function doesn't have to know what should happen next; whoever called it can decide, using try/except.

## assert or raise?

Both stop the program with an exception, but they are for different jobs:

* Use `assert` to check things that should never happen if *your own code* is correct. A failing assert means there is a bug to fix.
* Use `raise` for problems that can happen even when the code is correct: bad user input, a missing file, a malformed sequence. These are the errors that callers may want to catch and handle.

## Making your own kind of exception

Exceptions are classes, so you can make a new kind by inheriting from an existing one (Lecture 14 again). Often the class body needs nothing in it at all. The point is the name, which lets callers catch your error specifically.

In [30]:
class InvalidDNAError(ValueError):
  """ Raised when a sequence contains something other than A, C, G or T """
  pass

def validate_dna(seq):
  """ Raises an InvalidDNAError if seq is not a valid DNA sequence """
  for i in range(len(seq)):
    if seq[i].upper() not in "ACGT":
      raise InvalidDNAError(f"unexpected character '{seq[i]}' at position {i}")

for seq in ["GATTACA", "GATXACA", "ACGU"]:
  try:
    validate_dna(seq)
    print(seq, "is fine")
  except InvalidDNAError as e:
    print(seq, "is not valid DNA:", e)

GATTACA is fine
GATXACA is not valid DNA: unexpected character 'X' at position 3
ACGU is not valid DNA: unexpected character 'U' at position 3


Because `InvalidDNAError` inherits from `ValueError`, code that says `except ValueError:` will catch it as well.

You can also use raise to "rethrow" an exception you have caught. This is useful when you want to note that something went wrong, but still let the error carry on up to whoever called you:

In [31]:
try:  # This is a contrived example
  assert 1 == 2
except AssertionError as e:
  print("Got an error", e)
  raise  # This "rethrows" the exception we caught (note: we don't have to tell Python which exception)

Got an error 


AssertionError: 

Note: There is a lot more to say about exceptions and writing fault tolerant code, but hopefully this summary is a good start!

# Putting the two together: testing that code fails correctly

Back to the design question from the bug hunt: what should `gc_content("GATXACA")` do? Returning a number would be quietly wrong, so let's have it raise our new exception.

A deliberate error is part of a function's behavior, so it deserves a test like any other. `assertRaises` passes only if the code inside its `with` block raises the exception you name:

In [32]:
def gc_content(seq):
  """ Returns the fraction of the bases in a DNA sequence that are G or C.
  Raises an InvalidDNAError if the sequence contains anything else.
  """
  validate_dna(seq)
  if len(seq) == 0:
    return 0
  seq = seq.upper()
  return (seq.count("G") + seq.count("C")) / len(seq)

class GCContentTest(unittest.TestCase):
  """ Tests gc_content() """

  def test_typical(self):
    self.assertAlmostEqual(gc_content("GGCC"), 1.0)
    self.assertAlmostEqual(gc_content("ATAT"), 0.0)
    self.assertAlmostEqual(gc_content("GATTACA"), 2/7)

  def test_lowercase(self):
    self.assertAlmostEqual(gc_content("ggcc"), 1.0)

  def test_empty(self):
    self.assertEqual(gc_content(""), 0)

  def test_invalid_characters(self):
    with self.assertRaises(InvalidDNAError):
      gc_content("GATXACA")
    with self.assertRaises(InvalidDNAError):
      gc_content("ACGU")  # RNA, not DNA

unittest.main(argv=['first-arg-is-ignored', 'GCContentTest'], exit=False)

....
----------------------------------------------------------------------
Ran 4 tests in 0.004s

OK


# Challenge 3

In [ ]:
# Write code that uses exception handling and the get_age function defined above
# to probe a user for a valid age, repeating the prompt until a valid age is given.

# Cheat sheet

**Unit testing**

| You write... | ...and get |
|---|---|
| `class MyTest(unittest.TestCase):` | a group of tests |
| `def test_something(self):` | one test (the name must start with `test`) |
| `def setUp(self):` / `def tearDown(self):` | code run before / after every test |
| `self.assertEqual(a, b)`, `self.assertTrue(x)`, `self.assertAlmostEqual(a, b)` | checks; a failed check is reported as `F` |
| `with self.assertRaises(SomeError):` | a check that the block raises `SomeError` |
| `unittest.main(argv=['first-arg-is-ignored'], exit=False)` | all the tests in the notebook run |

**Exceptions**

| You write... | ...and get |
|---|---|
| `try:` ... `except ValueError:` | the except block runs if the try block raises a ValueError |
| `except ValueError as e:` | the same, with the exception object available as `e` |
| `except (TypeError, ValueError):` | one handler for several types |
| `except:` | catches everything, including your typos; avoid |
| `else:` | runs only if the try block raised nothing |
| `finally:` | runs whatever happened |
| `raise ValueError("message")` | an exception of your own |
| `raise` (inside an except block) | the caught exception, rethrown |
| `class MyError(Exception): pass` | a new kind of exception |

# Reading

* Open book Chapter 19: (exceptions)
http://openbookproject.net/thinkcs/python/english3e/exceptions.html

# Homework

* Go to Canvas and complete the lecture quiz, which involves completing each challenge problem
* Zybooks Reading 16

# Practice Problems

In [ ]:
"""
Practice Problem 1: File Word Counter with Error Handling

Write a class FileWordCounter that:
- Takes a filename in its constructor
- Has a method count_words() that:
    - Returns the number of words in the file
    - Handles file not found errors by returning 0 (this is weird behavior, but for the purposes of the test)
    - Closes the file properly even if there's an error
    - Uses try/except/(possibly) finally
- Has a method get_word_frequency(word) that:
    - Returns how many times a word appears
    - Is case-insensitive
    - Ignores punctuation attached to a word, so that "test." counts as "test"

Write your class here:
"""

# Your code here


# Tests for Problem 1
def test_word_counter():
  # Create a test file
  with open("test.txt", "w") as f:
    f.write("This is a test file.\nThis is only a test.")

  counter = FileWordCounter("test.txt")
  assert counter.count_words() == 10
  assert counter.get_word_frequency("test") == 2
  assert counter.get_word_frequency("THIS") == 2
  assert counter.get_word_frequency("nonexistent") == 0

  # Test error handling case where file doesn't exist
  counter = FileWordCounter("nonexistent.txt")
  assert counter.count_words() == 0

  print("All word counter tests passed!")

test_word_counter()

In [ ]:
import unittest

"""
Practice Problem 2: Temperature Converter with Unit Tests

Write a class TemperatureConverter that:
- Converts between Celsius and Fahrenheit
- Has methods:
    - to_celsius(fahrenheit)
    - to_fahrenheit(celsius)
- Raises ValueError for temperatures below absolute zero (-273.15°C or -459.67°F)
- Here assume: °C = (°F - 32) x (5/9)

Write your class here:
"""

# Your code here


# Tests for Problem 2
class TestTemperatureConverter(unittest.TestCase):
  def setUp(self):
    self.converter = TemperatureConverter()

  def test_to_celsius(self):
    self.assertAlmostEqual(self.converter.to_celsius(32), 0)
    self.assertAlmostEqual(self.converter.to_celsius(212), 100)
    with self.assertRaises(ValueError):
      self.converter.to_celsius(-500)

  def test_to_fahrenheit(self):
    self.assertAlmostEqual(self.converter.to_fahrenheit(0), 32)
    self.assertAlmostEqual(self.converter.to_fahrenheit(100), 212)
    with self.assertRaises(ValueError):
      self.converter.to_fahrenheit(-300)

unittest.main(argv=['first-arg-is-ignored', 'TestTemperatureConverter'], exit=False)

In [ ]:
import unittest

"""
Practice Problem 3: Password Strength Checker

Write a class PasswordStrengthChecker that:
- Has a method check_strength that takes a password string
- Raises custom exceptions for different password requirements:
    - TooShortError if less than 8 characters
    - NoUppercaseError if no uppercase letters
    - NoNumberError if no numbers
  (check the requirements in that order)
- Returns True if password meets all requirements

Write your class and exceptions here:
"""

# Define custom exceptions here
class TooShortError(Exception):
  pass

class NoUppercaseError(Exception):
  pass

class NoNumberError(Exception):
  pass

class PasswordStrengthChecker:
  # Your code here
  pass


# Tests for Problem 3
class TestPasswordStrengthChecker(unittest.TestCase):
  def setUp(self):
    self.checker = PasswordStrengthChecker()

  def test_too_short(self):
    with self.assertRaises(TooShortError):
      self.checker.check_strength("short")

  def test_no_uppercase(self):
    with self.assertRaises(NoUppercaseError):
      self.checker.check_strength("nouppercase123")

  def test_no_number(self):
    with self.assertRaises(NoNumberError):
      self.checker.check_strength("NoNumbers")

  def test_good_password(self):
    self.assertTrue(self.checker.check_strength("GoodPass123"))

unittest.main(argv=['first-arg-is-ignored', 'TestPasswordStrengthChecker'], exit=False)

In [ ]:
import unittest

"""
Practice Problem 4: Write the tests first

This time you write the tests as well as the code, in that order.

The function reverse_complement(seq) should:
- Return the reverse complement of a DNA sequence: reverse the sequence and swap
  A with T and C with G. For example reverse_complement("GATTACA") is "TGTAATC"
- Return its answer in upper case, whatever the case of the input
- Return "" for an empty sequence
- Raise a ValueError if the sequence contains any character other than A, C, G or T

Step 1: Write a test class ReverseComplementTest with at least four test methods,
        covering the four points above. Run it and watch every test fail.
Step 2: Write reverse_complement so that the tests pass.
"""

def reverse_complement(seq):
  # Your code here (step 2)
  pass

class ReverseComplementTest(unittest.TestCase):
  # Your tests here (step 1)
  pass


# This runs your tests and checks that there are at least four and that they all pass
suite = unittest.defaultTestLoader.loadTestsFromTestCase(ReverseComplementTest)
result = unittest.TextTestRunner().run(suite)
assert result.testsRun >= 4, "Write at least four test methods"
assert result.wasSuccessful(), "Some of your tests are failing"